# 04. Differential expression with DESeq2

このノートブックは、gene count matrixからDEGを検出する段階である。

**この段階が答える問い**

- ある条件で、どのgeneが基準群より上がったか、下がったか。
- その変化は統計的にどれくらい信頼できるか。

**入力**

- `results/counts/gene_counts.tsv`
- `metadata/samples.tsv`
- `metadata/contrasts.tsv`

**出力**

- `results/de/deseq2_<contrast_id>.csv`
- `results/de/deseq2_normalized_counts.tsv`
- `results/de/deseq2_vst_counts.tsv`
- `results/de/MA_<contrast_id>.pdf`


## DESeq2モデルの見取り図

DESeq2は、gene `g`、sample `s` のcountを負の二項分布で扱う。

`count_gs ~ NegativeBinomial(mean_gs, dispersion_g)`

`design_formula = ~ condition` は、発現量の違いを `condition` の違いとして推定する設定である。

batchを補正したい場合は、metadataに `batch` 列を作り、設定を `~ batch + condition` に変える。ただし、batchとconditionが完全に重なっている設計では補正できない。


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess

import pandas as pd

PROJECT_DIR = Path("/Users/yusuke_tateishi/Documents/RNA_seq").resolve()
CONFIG = json.loads((PROJECT_DIR / "config" / "analysis_config.json").read_text(encoding="utf-8"))
COUNTS_PATH = PROJECT_DIR / CONFIG["differential_expression"]["count_matrix_path"]
SAMPLES_PATH = PROJECT_DIR / CONFIG["samples_path"]
CONTRASTS_PATH = PROJECT_DIR / CONFIG["contrasts_path"]
DE_DIR = PROJECT_DIR / "results" / "de"
DE_DIR.mkdir(parents=True, exist_ok=True)

for path in [COUNTS_PATH, SAMPLES_PATH, CONTRASTS_PATH]:
    print(path.relative_to(PROJECT_DIR), "exists:", path.exists())


## 入力表の確認

count matrixのサンプル列とmetadataの `sample_id` が一致している必要がある。


In [ ]:
if not COUNTS_PATH.exists():
    raise FileNotFoundError(f"Count matrix not found: {COUNTS_PATH}. Run notebook 02 first.")

counts = pd.read_csv(COUNTS_PATH, sep="\t", nrows=5)
samples = pd.read_csv(SAMPLES_PATH, sep="\t")
contrasts = pd.read_csv(CONTRASTS_PATH, sep="\t")

count_sample_columns = [column for column in counts.columns if column != "gene_id"]
print("count matrix sample columns:", count_sample_columns)
print("metadata sample IDs:", list(samples["sample_id"]))
print("missing in counts:", sorted(set(samples["sample_id"]) - set(count_sample_columns)))
print("extra in counts:", sorted(set(count_sample_columns) - set(samples["sample_id"])))
display(contrasts)


## DESeq2実行コマンド

実際のDESeq2処理は `scripts/deseq2_differential_expression.R` に置いている。ノートブックは入力確認と実行管理に集中する。


In [ ]:
DESEQ2_SCRIPT = PROJECT_DIR / "scripts" / "deseq2_differential_expression.R"

deseq2_command = [
    "Rscript",
    str(DESEQ2_SCRIPT),
    f"--counts={COUNTS_PATH}",
    f"--metadata={SAMPLES_PATH}",
    f"--contrasts={CONTRASTS_PATH}",
    f"--outdir={DE_DIR}",
    f"--design={CONFIG['differential_expression']['design_formula']}",
    f"--reference={CONFIG['differential_expression']['reference_condition']}",
    f"--min-count={CONFIG['differential_expression']['min_count']}",
    f"--min-samples={CONFIG['differential_expression']['min_samples']}",
    f"--alpha={CONFIG['differential_expression']['alpha']}",
]

command_path = DE_DIR / "deseq2_command.txt"
command_path.write_text(" ".join(deseq2_command) + "\n", encoding="utf-8")
print("Command written to:", command_path.relative_to(PROJECT_DIR))
print(" ".join(deseq2_command))


In [ ]:
RUN_DESEQ2 = False

if RUN_DESEQ2:
    rscript = shutil.which("Rscript")
    if rscript is None:
        raise RuntimeError("Rscript was not found. Activate the rna-seq environment first.")
    deseq2_command[0] = rscript
    subprocess.run(deseq2_command, check=True)
else:
    print("Set RUN_DESEQ2 = True after sample QC looks acceptable.")


## 結果の読み方

DESeq2結果の主な列である。

- `baseMean`: そのgeneの平均的な発現量。
- `log2FoldChange`: test condition / reference condition のlog2比。
- `pvalue`: 統計検定のp値。
- `padj`: 多重検定補正後のp値。通常はこちらを重視する。
- `direction`: このプロジェクトのRスクリプトが付けた簡易ラベル。


In [ ]:
result_files = sorted(DE_DIR.glob("deseq2_*.csv"))
result_files = [path for path in result_files if path.name not in {"deseq2_normalized_counts.tsv", "deseq2_vst_counts.tsv"}]

summaries = []
for path in result_files:
    de = pd.read_csv(path)
    summaries.append(
        {
            "file": path.name,
            "rows": len(de),
            "padj_lt_alpha": int((de["padj"] < CONFIG["differential_expression"]["alpha"]).sum()),
            "up": int((de["direction"] == "up").sum()) if "direction" in de.columns else None,
            "down": int((de["direction"] == "down").sum()) if "direction" in de.columns else None,
        }
    )

if summaries:
    display(pd.DataFrame(summaries))
else:
    print("No DESeq2 result files yet. Run DESeq2 first.")


**よくある間違い**

- `padj` ではなく、生の `pvalue` だけで判断する。
- PCAで外れ値があるのに、そのままDESeq2へ進む。
- `reference_condition` を確認せず、log2 fold changeの向きを誤解する。

**小さい練習**

`Oxi_2h_vs_Non` の `log2FoldChange > 0` は「Oxi_2hでNonより高い」という意味である。この向きを自分の言葉で確認しよう。
